# Exercise 1

## Imports

In [52]:
using LinearAlgebra, Statistics
using MAT
using Dates

include("bidiag2.jl")
include("svdr.jl")
include("TregsRLooCV.jl")


TregsRLooCV (generic function with 1 method)

## 1.

In [53]:
# Taken from MMDPR7more
function PLS_nip(X, y; mc = 2)
# β₀, Β, T, W, P, q = PLS_nip(X,y; mc = 3)
m,n = size(X)
mc  = min(mc, min(n,m)-1) # Assure that the number of extracted components is consistent with the size of the problem.
T   = zeros(m,mc); W   = zeros(n,mc); P   = zeros(n,mc)
q   = zeros(1,mc);        # - the regression coeffs for the PLS-scores.
x̄   = mean(X, dims=1)     # - row vector of the X-column mean values.
ȳ   = mean(y, dims=1)[1]  # - the mean of the response values y.
y   = y.- ȳ;              # - the centered response vector.
X   = X.- x̄;              # - the centered X-data
for a = 1:mc
    w = X'y;  w = w/norm(w);  W[:,a] = w;
    t = X*w;  t = t/norm(t);  T[:,a] = t;
# ------------------- Deflate X and y ----------------------
    P[:,a] = X't;         X = X - t*P[:,a]';
    q[a]   = (y't)[1];    y = y - q[a].*t;
end
# ---------- Calculate regression coefficients -------------
Β = cumsum((W/triu(P'W)).*q, dims = 2); # the PLS-regression coeffs for the X-data based on up to mc components.
#Β  = cumsum((W/Bidiagonal(P'W, :U)).*q, dims = 2); # the PLS-regression coeffs for the X-data based on up to mc components.
β₀ = ȳ .- x̄*Β;            # the correspondning constant terms for the PLS-models.
return β₀, Β, T, W, P, q;
end

PLS_nip (generic function with 1 method)

In [54]:
function bidiag2(X, y; mc = 2)

    m,n = size(X)
    mc  = min(mc, min(n,m)-1)

    T = zeros(m,mc)
    W = zeros(n,mc)
    P = zeros(n,mc)
    β = zeros(n,mc)
    q = zeros(1,mc)

    x̄ = mean(X, dims=1)
    ȳ = mean(y, dims=1)

    X = X .- x̄
    y = y .- ȳ

    # --- bidiagonalization ---
    w = X' * y
    w = w / norm(w)
    W[:,1] = w

    t = X * w
    p = norm(t)
    t = t / p
    T[:,1] = t

    P[:,1] = X' * t
    q[1] = (y' * t)[1]

    d = w / p
    β[:,1] = q[1] * d

    # --- Continue ---
    for a = 2:mc

        w = X' * t - p * w
        w = w - W[:,1:a-1]*(W[:,1:a-1]' * w)
        θ = norm(w)
        w = w / θ
        W[:,a] = w

        t = X * w - θ * t
        t = t - T[:,1:a-1]*(T[:,1:a-1]' * t)
        p = norm(t)
        t = t / p
        T[:,a] = t

        P[:,a] = X' * t
        q[a] = (y' * t)[1]

        d = (w - θ*d)/p
        β[:,a] = β[:,a-1] + q[a] * d
    end

    β₀ = ȳ .- x̄ * β

    return β₀, β, T, W, P, q
end

bidiag2 (generic function with 1 method)

In [55]:
function PLSHY(X, y; mc = 2)

    m,n = size(X)
    mc  = min(mc, min(n,m)-1)

    T = zeros(m,mc)
    W = zeros(n,mc)
    P = zeros(n,mc)
    β = zeros(n,mc)
    q = zeros(1,mc)

    x̄ = mean(X, dims=1)
    ȳ = mean(y, dims=1)

    X = X .- x̄
    y = y .- ȳ

    d = zeros(n)

    for a = 1:mc

        w = X' * y
        w = w / norm(w)
        W[:,a] = w

        t = X * w
        t = t / norm(t)
        T[:,a] = t

        P[:,a] = X' * t
        q[a] = (y' * t)[1]

        X = X - t * P[:,a]'
        y = y - q[a] * t

        if a == 1
            d = w
            β[:,a] = q[a] * d
        else
            d = w - P[:,a-1] * (P[:,a-1]' * w)
            β[:,a] = β[:,a-1] + q[a] * d
        end
    end

    β₀ = ȳ .- x̄ * β

    return β₀, β, T, W, P, q
end

PLSHY (generic function with 1 method)

## 2.

In [56]:
# Loading Data
data = matread("spectra.mat")

X = data["NIR"]
y = vec(data["octane"])

60-element Vector{Float64}:
 85.3
 85.25
 88.45
 83.4
 87.9
 85.5
 88.9
 88.3
 88.7
 88.45
  ⋮
 87.6
 88.35
 85.1
 85.1
 84.7
 87.2
 86.6
 89.6
 87.1

In [57]:
mc = 10

β₀_n, β_n, T_n, W_n, P_n, q_n = PLS_nip(X, y, mc=mc)
β₀_b, β_b, T_b, W_b, P_b, q_b = bidiag2(X, y, mc=mc)
β₀_h, β_h, T_h, W_h, P_h, q_h = PLSHY(X, y, mc=mc)

([77.25848062527871 69.0500724143939 … 72.58232084294062 72.50453354267725], [-0.030190087462644545 -0.006935965912778487 … 0.015065663568405455 0.014014225190819542; -0.019148369693699086 0.029159237490900155 … 0.0356387016935614 0.03430693123968637; … ; -0.06564300344311907 -0.27578874284250643 … -0.824078076675605 -0.7309518336490479; 0.1977132423989957 0.5940968593710547 … 0.3513406569655062 0.2796965356629069], [-0.04012941725727484 -0.17638494147295483 … 0.07059414086260983 -0.14714032711030478; -0.34081007576062705 -0.003288551346646735 … 0.06632731512361316 -0.07264269586654976; … ; 0.19870690023016963 0.04233121306267645 … 0.014524801745465688 0.16230558334233214; 0.04949060105959129 -0.07333137043246676 … 0.027306701556812772 -0.18753411659952351], [-0.0045478150963036675 0.014675759194317187 … -0.04470033686754882 -0.003965574582588681; -0.002884497929009317 0.0167119473681744 … -0.04132363367740054 -0.005082607145431979; … ; -0.009888419250017641 -0.02560775798831302 … -0.0

In [58]:
# Comparing
println("β difference (bidiag2 vs PLS_nip): ", norm(β_b - β_n))
println("β difference (PLSHY vs PLS_nip): ", norm(β_h - β_n))
println("T difference (bidiag2 vs PLS_nip): ", norm(T_b - T_n))
println("T difference (PLSHY vs PLS_nip): ", norm(T_h - T_n))
println("W difference (bidiag2 vs PLS_nip): ", norm(W_b - W_n))
println("W difference (PLSHY vs PLS_nip): ", norm(W_h - W_n))

β difference (bidiag2 vs PLS_nip): 2.4972620285467624e-12
β difference (PLSHY vs PLS_nip): 59.829782164982355
T difference (bidiag2 vs PLS_nip): 4.47213595499958
T difference (PLSHY vs PLS_nip): 0.0
W difference (bidiag2 vs PLS_nip): 4.47213595499958
W difference (PLSHY vs PLS_nip): 0.0


The regression coefficients from bidiag2 and PLS_nip are almost identical, with a very small difference, which points that these two methods produce the same model. However, the PLSHY method gives a much larger difference in the regression coefficients, telling us that something is different in that implementation.

For the scores (T) and weights (W), bidiag2 differs from PLS_nip, while PLSHY matches PLS_nip exactly. This is not unexpected, since the latent variables in PLS are not unique and can differ by scaling or rotation without affecting the final regression model.

## 3.

In [59]:
# generating random dataset
m = 300
n = 1000

X_rand = rand(m, n)
y_rand = rand(m)

300-element Vector{Float64}:
 0.9784779855151853
 0.5101241255556576
 0.07424836804103985
 0.9416918871593528
 0.13134016963269723
 0.09107118646087708
 0.770364741374813
 0.28928824590790236
 0.03005160287183062
 0.21594239659567305
 ⋮
 0.4558258334294377
 0.1047977634873708
 0.25670595685940467
 0.88233692723099
 0.49097273966755794
 0.8142322897248421
 0.4225020252487922
 0.05565551828416038
 0.3651385745319792

In [60]:
function loocv_time(method, X, y, mc)
    m = size(X,1)
    t_start = time()

    for i in 1:m
        idx = setdiff(1:m, [i])

        if method == "nip"
            PLS_nip(X[idx,:], y[idx], mc=mc)
        elseif method == "bidiag2"
            bidiag2(X[idx,:], y[idx], mc=mc)
        elseif method == "plshy"
            PLSHY(X[idx,:], y[idx], mc=mc)
        end
    end

    return time() - t_start
end

loocv_time (generic function with 1 method)

In [61]:
mc = 10

t_n = loocv_time("nip", X_rand, y_rand, mc)
t_b = loocv_time("bidiag2", X_rand, y_rand, mc)
t_h = loocv_time("plshy", X_rand, y_rand, mc)

println("PLS_nip time: ", t_n)
println("bidiag2 time: ", t_b)
println("PLSHY time: ", t_h)

PLS_nip time: 7.452999830245972
bidiag2 time: 0.9539999961853027
PLSHY time: 6.766999959945679


PLS_nip is generally slower because it relies on iterative deflation and repeated calculations in each step. In contrast, bidiag2 is more efficient due to its bidiagonalization approach, which reduces the computational cost. The PLSHY method is also designed to improve numerical stability and efficiency, although its performance depends on the implementation.